<a href="https://colab.research.google.com/github/BorysZhyhalo/goit-rdb-hw-03/blob/main/Setup_PostgreSQL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Примусове встановлення pgserver для Python 3.13, ігноруючи обмеження версій пакета
!pip install -q pgserver==0.1.4 --python-version 3.12 --only-binary=:all: --target=/usr/local/lib/python3.13/dist-packages

!pip install -q sqlalchemy psycopg2-binary pandas

# ===================================================
# КОД UPDATED:
# ===================================================
import pgserver
from sqlalchemy import create_engine, text

# Стартує PostgreSQL у /tmp/pg_data
pg = pgserver.get_server('/tmp/pg_data')
engine = create_engine(pg.get_uri())

# Перевірка з'єднання
with engine.connect() as conn:
    version = conn.execute(text("SELECT version()")).scalar()
    print("\n--- УСПІШНЕ ПІДКЛЮЧЕННЯ ДО БД! ---")
    print(version)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 48.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 155.6/155.6 kB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 5.3 MB/s eta 0:00:00

--- УСПІШНЕ ПІДКЛЮЧЕННЯ ДО БД! ---
PostgreSQL 16.2 on x86_64-pc-linux-gnu, compiled by gcc (GCC) 10.2.1 20210130 (Red Hat 10.2.1-11), 64-bit


In [ ]:
# Виконуємо через engine.connect() (так само як і кожен подальший запит):
import pandas as pd
df = pd.read_sql("""
    SELECT
        current_database() AS db,
        current_user       AS usr,
        version()          AS pg_version
""", engine)
print(df)


         db       usr                                         pg_version
0  postgres  postgres  PostgreSQL 16.2 on x86_64-pc-linux-gnu, compil...


In [ ]:
import urllib.request
import sqlalchemy

# Public Northwind для PostgreSQL — MIT License
# Repo: <https://github.com/pthom/northwind_psql>
# Файл: northwind.sql (~62 KB, ~3300 рядків, 13 таблиць + дані)
url = "<https://raw.githubusercontent.com/pthom/northwind_psql/master/northwind.sql>"
schema_sql = urllib.request.urlopen(url).read().decode('utf-8')

# pgserver / psycopg приймають multi-statement strings через .execute()
# напряму. Виконуємо схему атомарно у межах однієї транзакції.
# Cave: уникаємо sql.split(';'), бо наївний split не парсить ';'
# усередині текстових літералів (напр. у INSERT ... VALUES
# ('Soft drinks; coffees', ...)) — і тихо зламає завантаження.
with engine.connect() as conn:
    conn.execute(sqlalchemy.text(schema_sql))
    conn.commit()

# Перевірка:
import pandas as pd
df = pd.read_sql("SELECT * FROM categories LIMIT 5", engine)
print(df)


   category_id   category_name  \
0            1       Beverages   
1            2      Condiments   
2            3     Confections   
3            4  Dairy Products   
4            5  Grains/Cereals   

                                         description picture  
0        Soft drinks, coffees, teas, beers, and ales      []  
1  Sweet and savory sauces, relishes, spreads, an...      []  
2                Desserts, candies, and sweet breads      []  
3                                            Cheeses      []  
4                Breads, crackers, pasta, and cereal      []  


In [ ]:
# 1. Інсталюємо необхідні бібліотеки
!pip install duckdb duckdb-engine ipython-sql -q

# 2. Завантажуємо SQL-розширення
%load_ext sql

# 3. Підключаємося до тимчасової бази даних у пам'яті
%sql duckdb:///:memory:

import duckdb


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 16.8 MB/s eta 0:00:00


In [ ]:


# Запускаємо весь ваш SQL-код як один текстовий блок всередині duckdb.sql()
duckdb.sql("""
-- ============================================================
-- ПРИКЛАД 1НФ: неправильна таблиця
-- ============================================================
DROP TABLE IF EXISTS demo_1nf_students_bad;

CREATE TEMP TABLE demo_1nf_students_bad (
    student_id INTEGER,
    student_name TEXT,
    phones TEXT
);

INSERT INTO demo_1nf_students_bad (student_id, student_name, phones)
VALUES
    (1, 'Anna', '111-111, 222-222'),
    (2, 'Petro', '333-333');

SELECT * FROM demo_1nf_students_bad;
""").show()  # .show() красиво надрукує таблицю в консоль


┌────────────┬──────────────┬──────────────────┐
│ student_id │ student_name │      phones      │
│   int32    │   varchar    │     varchar      │
├────────────┼──────────────┼──────────────────┤
│          1 │ Anna         │ 111-111, 222-222 │
│          2 │ Petro        │ 333-333          │
└────────────┴──────────────┴──────────────────┘



In [ ]:

# Запускаємо SQL-скрипт через DuckDB
duckdb.sql("""
-- ============================================================
-- ПРИКЛАД 1НФ: правильний варіант
-- Ідея: один телефон = один окремий рядок.
-- ============================================================

-- Спочатку видаляємо дочірню таблицю з телефонами.
-- Вона посилається на студентів, тому її видаляємо першою.
DROP TABLE IF EXISTS demo_1nf_student_phones;

-- Потім видаляємо таблицю студентів.
DROP TABLE IF EXISTS demo_1nf_students;

-- Таблиця студентів.
-- student_id — первинний ключ.
-- PRIMARY KEY означає: значення унікальне і не NULL.
CREATE TABLE demo_1nf_students (
    student_id   INTEGER PRIMARY KEY,
    student_name TEXT NOT NULL
);

-- Таблиця телефонів.
-- Тут кожен телефон зберігається окремим рядком.
CREATE TABLE demo_1nf_student_phones (
    student_id INTEGER NOT NULL,
    phone      TEXT NOT NULL,

    -- Один і той самий телефон не повинен дублюватися
    -- для одного й того самого студента.
    PRIMARY KEY (student_id, phone),

    -- FOREIGN KEY означає:
    -- student_id у цій таблиці має існувати
    -- в таблиці demo_1nf_students.
    FOREIGN KEY (student_id)
        REFERENCES demo_1nf_students(student_id)
);

-- Додаємо студентів.
INSERT INTO demo_1nf_students
    (student_id, student_name)
VALUES
    (1, 'Anna'),
    (2, 'Petro');

-- Додаємо телефони.
-- У Anna два телефони, тому для неї два рядки.
INSERT INTO demo_1nf_student_phones
    (student_id, phone)
VALUES
    (1, '111-111'),
    (1, '222-222'),
    (2, '333-333');
""")

# Виводимо фінальний результат об'єднання (JOIN) на екран
print("--- ОБ'ЄДНАНА ТАБЛИЦЯ (JOIN) ---")
duckdb.sql("""
SELECT
    s.student_id,
    s.student_name,
    p.phone
FROM demo_1nf_students AS s
JOIN demo_1nf_student_phones AS p
    ON s.student_id = p.student_id
ORDER BY s.student_id, p.phone;
""").show()


--- ОБ'ЄДНАНА ТАБЛИЦЯ (JOIN) ---
┌────────────┬──────────────┬─────────┐
│ student_id │ student_name │  phone  │
│   int32    │   varchar    │ varchar │
├────────────┼──────────────┼─────────┤
│          1 │ Anna         │ 111-111 │
│          1 │ Anna         │ 222-222 │
│          2 │ Petro        │ 333-333 │
└────────────┴──────────────┴─────────┘



In [ ]:
duckdb.sql("""
-- ============================================================
-- ПРИКЛАД 2НФ: неправильна таблиця
-- Проблема:
-- PK = (student_id, course_id)
-- Але student_name залежить тільки від student_id.
-- А course_name залежить тільки від course_id.
-- ============================================================

DROP TABLE IF EXISTS demo_2nf_enrollments_bad;

CREATE TABLE demo_2nf_enrollments_bad (
    student_id   INTEGER,
    course_id    INTEGER,
    student_name TEXT,
    course_name  TEXT,
    grade        INTEGER,

    -- Композитний первинний ключ:
    -- один студент може бути записаний на багато курсів,
    -- і один курс може мати багато студентів.
    -- Але одна пара student_id + course_id не повинна дублюватися.
    PRIMARY KEY (student_id, course_id)
);

-- Додаємо приклади.
INSERT INTO demo_2nf_enrollments_bad
    (student_id, course_id, student_name, course_name, grade)
VALUES
    (1, 10, 'Anna',  'SQL Basics', 95),
    (1, 20, 'Anna',  'Python ML',  88),
    (2, 10, 'Petro', 'SQL Basics', 76);
""")

# Виводимо вміст неправильної таблиці на екран
print("--- НЕПРАВИЛЬНА ТАБЛИЦЯ (ПОРУШЕННЯ 2НФ) ---")
duckdb.sql("SELECT * FROM demo_2nf_enrollments_bad;").show()

--- НЕПРАВИЛЬНА ТАБЛИЦЯ (ПОРУШЕННЯ 2НФ) ---
┌────────────┬───────────┬──────────────┬─────────────┬───────┐
│ student_id │ course_id │ student_name │ course_name │ grade │
│   int32    │   int32   │   varchar    │   varchar   │ int32 │
├────────────┼───────────┼──────────────┼─────────────┼───────┤
│          1 │        10 │ Anna         │ SQL Basics  │    95 │
│          1 │        20 │ Anna         │ Python ML   │    88 │
│          2 │        10 │ Petro        │ SQL Basics  │    76 │
└────────────┴───────────┴──────────────┴─────────────┴───────┘



In [ ]:
duckdb.sql("""
-- ============================================================
-- ПРИКЛАД 2НФ: правильний варіант
-- Ідея:
-- студенти окремо,
-- курси окремо,
-- записи студентів на курси окремо.
-- ============================================================

-- Спочатку видаляємо таблицю-зв'язку,
-- бо вона посилається на студентів і курси.
DROP TABLE IF EXISTS demo_2nf_enrollments;

-- Потім видаляємо довідникові таблиці.
DROP TABLE IF EXISTS demo_2nf_students;
DROP TABLE IF EXISTS demo_2nf_courses;

-- Таблиця студентів.
-- Тут зберігаються тільки факти про студентів.
CREATE TABLE demo_2nf_students (
    student_id   INTEGER PRIMARY KEY,
    student_name TEXT NOT NULL
);

-- Таблиця курсів.
-- Тут зберігаються тільки факти про курси.
CREATE TABLE demo_2nf_courses (
    course_id   INTEGER PRIMARY KEY,
    course_name TEXT NOT NULL
);

-- Таблиця запису на курси.
-- Тут зберігається тільки факт:
-- який студент записаний на який курс і яку оцінку отримав.
CREATE TABLE demo_2nf_enrollments (
    student_id INTEGER NOT NULL,
    course_id  INTEGER NOT NULL,
    grade      INTEGER,

    -- Композитний ключ:
    -- одна пара студент + курс не повинна повторюватися.
    PRIMARY KEY (student_id, course_id),

    -- student_id має існувати в таблиці студентів.
    FOREIGN KEY (student_id)
        REFERENCES demo_2nf_students(student_id),

    -- course_id має існувати в таблиці курсів.
    FOREIGN KEY (course_id)
        REFERENCES demo_2nf_courses(course_id),

    -- Оцінка має бути від 0 до 100.
    CHECK (grade >= 0 AND grade <= 100)
);

-- Додаємо студентів.
INSERT INTO demo_2nf_students
    (student_id, student_name)
VALUES
    (1, 'Anna'),
    (2, 'Petro');

-- Додаємо курси.
INSERT INTO demo_2nf_courses
    (course_id, course_name)
VALUES
    (10, 'SQL Basics'),
    (20, 'Python ML');

-- Додаємо записи на курси.
-- Тут уже немає student_name і course_name,
-- бо вони зберігаються у своїх таблицях.
INSERT INTO demo_2nf_enrollments
    (student_id, course_id, grade)
VALUES
    (1, 10, 95),
    (1, 20, 88),
    (2, 10, 76);
""")

# Виводимо "людський" результат через JOIN на екран
print("--- ПРАВИЛЬНА СТРУКТУРА (2НФ) ЧЕРЕЗ JOIN ---")
duckdb.sql("""
SELECT
    s.student_name,
    c.course_name,
    e.grade
FROM demo_2nf_enrollments AS e
JOIN demo_2nf_students AS s
    ON e.student_id = s.student_id
JOIN demo_2nf_courses AS c
    ON e.course_id = c.course_id
ORDER BY s.student_name, c.course_name;
""").show()

--- ПРАВИЛЬНА СТРУКТУРА (2НФ) ЧЕРЕЗ JOIN ---
┌──────────────┬─────────────┬───────┐
│ student_name │ course_name │ grade │
│   varchar    │   varchar   │ int32 │
├──────────────┼─────────────┼───────┤
│ Anna         │ Python ML   │    88 │
│ Anna         │ SQL Basics  │    95 │
│ Petro        │ SQL Basics  │    76 │
└──────────────┴─────────────┴───────┘



In [ ]:
duckdb.sql("""
-- ============================================================
-- ПРИКЛАД 3НФ: неправильна таблиця
-- Проблема:
-- department_phone залежить від department_name,
-- а не безпосередньо від student_id (транзитивна залежність).
-- ============================================================

-- Видаляємо таблицю, якщо вона вже існує.
DROP TABLE IF EXISTS demo_3nf_students_bad;

-- Створюємо таблицю студентів, у якій помилково
-- зберігаємо ще й дані про кафедру.
CREATE TABLE demo_3nf_students_bad (
    student_id       INTEGER PRIMARY KEY,
    student_name     TEXT NOT NULL,
    department_name  TEXT NOT NULL,
    department_phone TEXT NOT NULL
);

-- Додаємо студентів.
-- Anna і Petro навчаються на одній кафедрі.
-- Тому department_phone повторюється двічі.
INSERT INTO demo_3nf_students_bad
    (student_id, student_name, department_name, department_phone)
VALUES
    (1, 'Anna',  'Computer Science', '555-100'),
    (2, 'Petro', 'Computer Science', '555-100'),
    (3, 'Olena', 'Mathematics',      '555-200');
""")

# Виводимо вміст неправильної таблиці на екран
print("--- НЕПРАВИЛЬНА ТАБЛИЦЯ (ПОРУШЕННЯ 3НФ) ---")
duckdb.sql("SELECT * FROM demo_3nf_students_bad;").show()

--- НЕПРАВИЛЬНА ТАБЛИЦЯ (ПОРУШЕННЯ 3НФ) ---
┌────────────┬──────────────┬──────────────────┬──────────────────┐
│ student_id │ student_name │ department_name  │ department_phone │
│   int32    │   varchar    │     varchar      │     varchar      │
├────────────┼──────────────┼──────────────────┼──────────────────┤
│          1 │ Anna         │ Computer Science │ 555-100          │
│          2 │ Petro        │ Computer Science │ 555-100          │
│          3 │ Olena        │ Mathematics      │ 555-200          │
└────────────┴──────────────┴──────────────────┴──────────────────┘



In [ ]:
duckdb.sql("""
-- ============================================================
-- ПРИКЛАД 3НФ: правильний варіант
-- Ідея:
-- телефон кафедри зберігається в таблиці кафедр,
-- а студент тільки посилається на свою кафедру.
-- ============================================================

-- Спочатку видаляємо таблицю студентів,
-- бо вона посилається на таблицю кафедр.
DROP TABLE IF EXISTS demo_3nf_students;

-- Потім видаляємо таблицю кафедр.
DROP TABLE IF EXISTS demo_3nf_departments;

-- Таблиця кафедр.
-- Тут зберігаються тільки факти про кафедри.
CREATE TEMP TABLE demo_3nf_departments (
    department_id    INTEGER PRIMARY KEY,
    department_name  TEXT NOT NULL,
    department_phone TEXT NOT NULL
);

-- Таблиця студентів.
-- Тут зберігаються тільки факти про студентів.
-- department_id показує, до якої кафедри належить студент.
CREATE TEMP TABLE demo_3nf_students (
    student_id    INTEGER PRIMARY KEY,
    student_name  TEXT NOT NULL,
    department_id INTEGER NOT NULL,

    -- department_id має існувати в таблиці кафедр.
    FOREIGN KEY (department_id)
        REFERENCES demo_3nf_departments(department_id)
);

-- Додаємо кафедри.
INSERT INTO demo_3nf_departments
    (department_id, department_name, department_phone)
VALUES
    (1, 'Computer Science', '555-100'),
    (2, 'Mathematics',      '555-200');

-- Додаємо студентів.
-- Замість назви кафедри і телефону зберігаємо тільки department_id.
INSERT INTO demo_3nf_students
    (student_id, student_name, department_id)
VALUES
    (1, 'Anna',  1),
    (2, 'Petro', 1),
    (3, 'Olena', 2);

-- Перевіряємо результат.
-- JOIN дозволяє знову побачити студента разом із назвою
-- та телефоном кафедри.
SELECT
    s.student_name,
    d.department_name,
    d.department_phone
FROM demo_3nf_students AS s
JOIN demo_3nf_departments AS d
    ON s.department_id = d.department_id
ORDER BY s.student_name;""")


┌──────────────┬──────────────────┬──────────────────┐
│ student_name │ department_name  │ department_phone │
│   varchar    │     varchar      │     varchar      │
├──────────────┼──────────────────┼──────────────────┤
│ Anna         │ Computer Science │ 555-100          │
│ Olena        │ Mathematics      │ 555-200          │
│ Petro        │ Computer Science │ 555-100          │
└──────────────┴──────────────────┴──────────────────┘

In [ ]:
duckdb.sql("""
-- Змінимо телефон кафедри Computer Science.
-- Оновлюється тільки один рядок у таблиці кафедр.
UPDATE demo_3nf_departments
SET department_phone = '555-999'
WHERE department_id = 1;

-- Після JOIN усі студенти цієї кафедри
-- автоматично “бачать” новий телефон кафедри.
SELECT
    s.student_name,
    d.department_name,
    d.department_phone
FROM demo_3nf_students AS s
JOIN demo_3nf_departments AS d
    ON s.department_id = d.department_id
ORDER BY s.student_name;""")


┌──────────────┬──────────────────┬──────────────────┐
│ student_name │ department_name  │ department_phone │
│   varchar    │     varchar      │     varchar      │
├──────────────┼──────────────────┼──────────────────┤
│ Anna         │ Computer Science │ 555-999          │
│ Olena        │ Mathematics      │ 555-200          │
│ Petro        │ Computer Science │ 555-999          │
└──────────────┴──────────────────┴──────────────────┘

In [ ]:
duckdb.sql("""
-- ============================================================
-- Простий offline feature store у PostgreSQL
--
-- Ідея:
-- один користувач може мати кілька версій feature у часі.
-- valid_from = з якого моменту значення стало актуальним
-- valid_to   = до якого моменту значення було актуальним
-- NULL у valid_to означає: значення актуальне досі
-- ============================================================

-- Видаляємо таблицю, якщо вона вже існує.
DROP TABLE IF EXISTS demo_features;

-- Створюємо тимчасову таблицю з історією features.
CREATE TEMP TABLE demo_features (
    user_id    INTEGER NOT NULL,
    orders_30d INTEGER NOT NULL,
    valid_from TIMESTAMPTZ NOT NULL,
    valid_to   TIMESTAMPTZ,

    -- Один user_id може мати багато історичних версій.
    -- Тому ключ складається з user_id + valid_from.
    PRIMARY KEY (user_id, valid_from),

    -- Якщо valid_to задано, воно має бути пізніше за valid_from.
    CHECK (valid_to IS NULL OR valid_to > valid_from),

    -- Кількість замовлень не може бути від'ємною.
    CHECK (orders_30d >= 0)
);""")


In [ ]:
duckdb.sql("""
-- ============================================================
-- Історія orders_30d для user_id = 1
-- ============================================================

INSERT INTO demo_features
    (user_id, orders_30d, valid_from, valid_to)
VALUES
    -- З 1 січня до 15 січня користувач мав 2 замовлення за 30 днів.
    (1, 2,
     TIMESTAMPTZ '2025-01-01 00:00:00+00',
     TIMESTAMPTZ '2025-01-15 00:00:00+00'),

    -- З 15 січня значення змінилося: стало 5 замовлень.
    -- NULL у valid_to означає, що значення актуальне досі.
    (1, 5,
     TIMESTAMPTZ '2025-01-15 00:00:00+00',
     NULL);""")


In [ ]:
duckdb.sql("""
-- ============================================================
-- AS-OF SELECT:
-- беремо feature, яке було актуальним на 2025-01-10.
-- ============================================================

SELECT
    user_id,
    orders_30d
FROM demo_features
WHERE user_id = 1

  -- Feature вже мало існувати на потрібний момент.
  AND valid_from <= TIMESTAMPTZ '2025-01-10 00:00:00+00'

  -- Feature ще мало бути актуальним на потрібний момент.
  AND (
        valid_to > TIMESTAMPTZ '2025-01-10 00:00:00+00'
        OR valid_to IS NULL
      );
""")


┌─────────┬────────────┐
│ user_id │ orders_30d │
│  int32  │   int32    │
├─────────┼────────────┤
│       1 │          2 │
└─────────┴────────────┘

In [ ]:
duckdb.sql("""
-- ============================================================
-- Таблиця label-events
--
-- event_time = момент, на який ми формуємо training example
-- churned    = цільова змінна, тобто правильна відповідь
-- ============================================================

DROP TABLE IF EXISTS demo_labels;

CREATE TEMP TABLE demo_labels (
    user_id    INTEGER NOT NULL,
    event_time TIMESTAMPTZ NOT NULL,
    churned    BOOLEAN NOT NULL,

    PRIMARY KEY (user_id, event_time)
);

INSERT INTO demo_labels
    (user_id, event_time, churned)
VALUES
    (1, TIMESTAMPTZ '2025-01-10 00:00:00+00', FALSE),
    (1, TIMESTAMPTZ '2025-01-20 00:00:00+00', TRUE);
    """)


In [ ]:
duckdb.sql("""
-- ============================================================
-- Правильний training dataset:
-- кожен label-event отримує feature,
-- яке було актуальним саме на його event_time.
-- ============================================================

SELECT
    l.user_id,
    l.event_time,
    l.churned,
    f.orders_30d
FROM demo_labels AS l
JOIN demo_features AS f
    ON l.user_id = f.user_id

    -- Feature вже існувало на момент label-event.
    AND f.valid_from <= l.event_time

    -- Feature ще було актуальним на момент label-event.
    AND (
          f.valid_to > l.event_time
          OR f.valid_to IS NULL
        )
ORDER BY l.event_time;

""")

┌─────────┬──────────────────────────┬─────────┬────────────┐
│ user_id │        event_time        │ churned │ orders_30d │
│  int32  │ timestamp with time zone │ boolean │   int32    │
├─────────┼──────────────────────────┼─────────┼────────────┤
│       1 │ 2025-01-10 00:00:00+00   │ false   │          2 │
│       1 │ 2025-01-20 00:00:00+00   │ true    │          5 │
└─────────┴──────────────────────────┴─────────┴────────────┘

In [ ]:
duckdb.sql("""

DROP TABLE IF EXISTS t3_demo_users_staging;
DROP TABLE IF EXISTS t3_demo_users_clean;

CREATE TABLE t3_demo_users_staging (
    user_id_raw      TEXT,
    name_raw         TEXT,
    email_raw        TEXT,
    signup_date_raw  TEXT,
    country_raw      TEXT
);

CREATE TABLE t3_demo_users_clean (
    user_id      INTEGER PRIMARY KEY,
    name         TEXT NOT NULL,
    email        TEXT,
    signup_date  DATE,
    country      VARCHAR(2)
);

-- Реальний COPY у staging:
--
-- COPY t3_demo_users_staging
-- FROM '/srv/loads/users.csv'
-- WITH (FORMAT csv, HEADER true, ENCODING 'UTF8');

-- Демонстраційні дані
INSERT INTO t3_demo_users_staging VALUES
    ('1', ' Alice ', 'alice@example.com', '2025-03-01', 'UA'),
    ('2', 'Bob, Jr.', 'bob@example.com', '2025-03-02', 'DE'),
    ('3', 'Charlie', '', '2025-03-04', 'UK'),
    ('4', 'Diana', 'diana@example.com', '', 'UA');

INSERT INTO t3_demo_users_clean (
    user_id,
    name,
    email,
    signup_date,
    country
)
SELECT
    user_id_raw::INTEGER,
    TRIM(name_raw),
    NULLIF(TRIM(email_raw), ''),
    NULLIF(TRIM(signup_date_raw), '')::DATE,
    UPPER(TRIM(country_raw))::VARCHAR(2)
FROM t3_demo_users_staging;

SELECT *
FROM t3_demo_users_clean
ORDER BY user_id;


""")


┌─────────┬──────────┬───────────────────┬─────────────┬─────────┐
│ user_id │   name   │       email       │ signup_date │ country │
│  int32  │ varchar  │      varchar      │    date     │ varchar │
├─────────┼──────────┼───────────────────┼─────────────┼─────────┤
│       1 │ Alice    │ alice@example.com │ 2025-03-01  │ UA      │
│       2 │ Bob, Jr. │ bob@example.com   │ 2025-03-02  │ DE      │
│       3 │ Charlie  │ NULL              │ 2025-03-04  │ UK      │
│       4 │ Diana    │ diana@example.com │ NULL        │ UA      │
└─────────┴──────────┴───────────────────┴─────────────┴─────────┘

In [ ]:
duckdb.sql("""
DROP TABLE IF EXISTS t3_demo_user_features;

CREATE TABLE t3_demo_user_features (
    user_id  INTEGER PRIMARY KEY,
    age      INTEGER,
    income   NUMERIC(10, 2),
    region   TEXT
);

INSERT INTO t3_demo_user_features (user_id, age, income, region) VALUES
    (1, 25,    50000.00, 'EU'),
    (2, NULL,  NULL,     'US'),
    (3, 30,    NULL,     NULL),
    (4, NULL,  75000.00, 'EU');

-- У цьому прикладі 28 — умовна константа,
-- яка імітує заздалегідь пораховану медіану age.
SELECT
    user_id,
    COALESCE(age, 28)                    AS age_filled,
    COALESCE(income, 0::NUMERIC(10, 2))  AS income_filled,
    COALESCE(region, 'unknown')          AS region_filled
FROM t3_demo_user_features
ORDER BY user_id;


""")


┌─────────┬────────────┬───────────────┬───────────────┐
│ user_id │ age_filled │ income_filled │ region_filled │
│  int32  │   int32    │ decimal(10,2) │    varchar    │
├─────────┼────────────┼───────────────┼───────────────┤
│       1 │         25 │      50000.00 │ EU            │
│       2 │         28 │          0.00 │ US            │
│       3 │         30 │          0.00 │ unknown       │
│       4 │         28 │      75000.00 │ EU            │
└─────────┴────────────┴───────────────┴───────────────┘